In [1]:
import os
import sys
# Agregar el directorio raíz del proyecto al path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
    
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *   
from services.utilidades import *
from services.comprobar_netcdf import *
from services.guardar_puntos_eliminados import guardar_puntos_eliminados

In [2]:
# Recargar módulos locales (ejecutar esta celda cuando cambies código en services/)
import importlib
import sys

# Lista de módulos locales a recargar
modulos_locales = [
    'services.carga_de_datos',
    'services.DataFrame_maker',
    'services.prueba',
    'services.utilidades',
    'services.get_tspan',
    'services.polar2uv',
    'services.uv2polar'
]

for modulo in modulos_locales:
    if modulo in sys.modules:
        importlib.reload(sys.modules[modulo])
        print(f"✓ Recargado: {modulo}")
    else:
        print(f"- No cargado: {modulo}")

# Re-importar después de recargar
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *

print("\n¡Módulos locales recargados!")

✓ Recargado: services.carga_de_datos
✓ Recargado: services.DataFrame_maker
- No cargado: services.prueba
✓ Recargado: services.utilidades
✓ Recargado: services.get_tspan
✓ Recargado: services.polar2uv
✓ Recargado: services.uv2polar

¡Módulos locales recargados!


In [3]:
# %matplotlib tk

In [4]:
dict_de_boyas = {}

### 2. OLEAJE

In [5]:
# p1: Crudo unido men-realt
nombre = "BOT1-05-T50_pkl_C"
archivos_csv = "msj3_crudo_unido_realt_2026030800_2026060823.pkl"
ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_1.2\\2026\\06. Junio\\BOT1-05-T50\\DATOS"
df_pkl_crudo_realt = cargar_pickle_oleaje(ruta_a_carpeta, archivos_csv)
dict_de_boyas[nombre] = df_pkl_crudo_realt # Guardo el dataframe dentro del diccionario de boyas.
df_pkl_crudo_realt.head(2)

,tspan,Hs,Hmax,Tp,dir,vardir
0,2026-03-08 00:00:00,0.82,1.3,6.1,59.6,20.3
1,2026-03-08 01:00:00,0.82,1.3,6.3,58.0,20.4


In [6]:
# 1.5. p1: Cargar NETCDF de oleaje 
nombre = "BOT1-05-T50_NC"
nombre_de_archivo = "BOT1-05-T50-TRIAXYSG3-NS09310-Z0-REALT-20260308-20260608.nc"
ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_1.2\\2026\\06. Junio\\BOT1-05-T50\\DATOS"
df_nc, nc, DirSpec = cargar_nc_oleaje(ruta_a_carpeta, nombre_de_archivo)
dict_de_boyas[nombre] = df_nc # Guardo el dataframe dentro del diccionario de boyas.
df_nc.head(2)


,tspan,Hs,Hmax,Tp,dir,vardir
0,2026-03-08 00:00:00,0.82,1.3,6.1,59.6,20.3
1,2026-03-08 01:00:00,0.82,1.3,6.3,58.0,20.4


In [7]:
# Otras revisiones del NC:
# comprobar_nombre_nc
comprobar_nombre_nc(nombre_de_archivo, nc)
comprobar_porcentaje_de_variable(nc, ruta_a_carpeta, "BOT1-05-T50_datos_word_2026030800_2026060823.xlsx", variables="oleaje")

NAM correcto


,Variable,esperados,validados,porcentaje_validados
0,Hs,2232,2211.0,99.059140
1,Hm,2232,2211.0,99.059140
2,Tp,2232,2211.0,99.059140
3,dir,2232,2211.0,99.059140
4,vardir,2232,2211.0,99.059140
5,dirspec,2232,2015.0,90.277778


In [8]:
# Crear el diccionario de boyas con los dataframes cargados
df = crear_dataframe_de_boyas(dict_de_boyas)

Index(['tspan', 'Hs_BOT1-05-T50_pkl_C', 'Hmax_BOT1-05-T50_pkl_C',
       'Tp_BOT1-05-T50_pkl_C', 'dir_BOT1-05-T50_pkl_C',
       'vardir_BOT1-05-T50_pkl_C', 'Hs_BOT1-05-T50_NC', 'Hmax_BOT1-05-T50_NC',
       'Tp_BOT1-05-T50_NC', 'dir_BOT1-05-T50_NC', 'vardir_BOT1-05-T50_NC'],
      dtype='str')


In [9]:
app = InteractivePlotterTk(df)
# Iniciar el bucle de la aplicación
app.root.mainloop()

# NOTAS PENDIENTES:
# - Arreglar que al eliminar puntos para que no tenga límite el historial de deshacer
# - Arreglar para que solo se grafiquen las columnas elegidas por el usuario
# - Agregar una función que grafique las variables de Rap y Dir --> sería algo como que df.iloc[idx,col][prof]
# - Arreglar el caso de cuando el usuario elimina puntos de Rap o Dir colocar U y V como nans --> df.iloc[idx,col][prof] = np.nan
# - Agregar la funcionalidad de guardar el DataFrame corregido en un archivo .csv o .xlsx, pkl
# - Agregar la funcionalidad de devolver puntos eliminados (no deshacer la última acción, sino devolver puntos específicos)

In [10]:
# nombre_de_archivo = "borrar_puntos"
# ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_1.2\\2026\\06. Junio\\BOT1-05-T50\\DATOS"
# guardar_puntos_eliminados(app, ruta_a_carpeta, nombre_de_archivo)